In [1]:
#"Yolo modelini kullanabilmek için kutuphane kurulumları"
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.1 MB/s eta 0:00:00


In [10]:
import os

path = '/kaggle/input/datasets'
print(os.listdir(path))

for item in os.listdir(path):
    full = os.path.join(path, item)
    if os.path.isdir(full):
        print(f"\n{item}/", os.listdir(full)[:10])

['tudorhirtopanu']

tudorhirtopanu/ ['yolo-highvis-and-person-detection-dataset']


In [12]:
path = '/kaggle/input/datasets/tudorhirtopanu/yolo-highvis-and-person-detection-dataset'
print(os.listdir(path))
path = '/kaggle/input/datasets/tudorhirtopanu/yolo-highvis-and-person-detection-dataset/YOLO-HiVis-Data'
print(os.listdir(path))

['YOLO-HiVis-Data']
['labels', 'images']


In [13]:
import os, shutil, random

dataset_path = '/kaggle/input/datasets/tudorhirtopanu/yolo-highvis-and-person-detection-dataset/YOLO-HiVis-Data'
output_path = '/kaggle/working/dataset'
# eğitimde kullanılacak train val ve test klasörleri oluşturuldu
for split in ['train', 'val', 'test']:
    os.makedirs(f'{output_path}/{split}/images', exist_ok=True)
    os.makedirs(f'{output_path}/{split}/labels', exist_ok=True)

images = [f for f in os.listdir(f'{dataset_path}/images') if f.endswith('.jpg')]
random.seed(42)
random.shuffle(images)

# veri setinin %70 i eğitim, %20 si validasyon %10 u da test olarak ayırdık.
train_end = int(len(images) * 0.7)
val_end = int(len(images) * 0.9)

splits = {
    'train': images[:train_end],
    'val': images[train_end:val_end],
    'test': images[val_end:]
}

for split, file_list in splits.items():
    for img in file_list:
        label = img.replace('.jpg', '.txt')
        shutil.copy(f'{dataset_path}/images/{img}', f'{output_path}/{split}/images/{img}')
        shutil.copy(f'{dataset_path}/labels/{label}', f'{output_path}/{split}/labels/{label}')

for split in ['train', 'val', 'test']:
    count = len(os.listdir(f'{output_path}/{split}/images'))
    print(f'{split}: {count} görüntü')

train: 5555 görüntü
val: 1588 görüntü
test: 794 görüntü


In [14]:
yaml_content = """
train: /kaggle/working/dataset/train/images
val: /kaggle/working/dataset/val/images
test: /kaggle/working/dataset/test/images

nc: 2
names: ['person', 'high-vis']
"""

with open('/kaggle/working/dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print("data.yaml oluştu")

data.yaml oluşturuldu!


In [15]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/kaggle/working/dataset/data.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    name='hivis_deneme_1'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=

In [16]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/runs/detect/hivis_deneme_1/weights/best.pt')

results = model.val(data='/kaggle/working/dataset/data.yaml', split='test')

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1103.1±405.1 MB/s, size: 39.1 KB)
val: Scanning /kaggle/working/dataset/test/labels... 794 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 794/794 1.3Kit/s 0.6s0.0ss
val: New cache created: /kaggle/working/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 7.1it/s 7.0s0.1s
                   all        794       2580      0.863       0.83      0.889      0.677
                person        546       1283      0.837      0.779      0.843      0.656
              high-vis        739       1297      0.889       0.88      0.935      0.698
Speed: 0.9ms preprocess, 3.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
